# Lorenz 96 data assimilation demo: true-solver burn-in + DM forward + observation guidance

Workflow:

1. **Burn-in** — true L96 solver spin-up.
2. **Load model** — conditional diffusion checkpoint (1D U-Net) + norm stats.
3. **Initialize ensemble** — perturbed histories for forecast / analysis.
4. **Observation + EDM settings** — configure mask and sampler.
5. **DA cycles** — ensemble forecast (no obs) vs **non-gradient** observation-guided analysis (`edm_sampler_obs`).

Same guidance as `scripts/SQG/da_ensf_dm_demo.ipynb` (no DPS).

Set `NETWORK_PKL`, `K`, `HISTORY`, and **`NE`** in the config cell.


In [ ]:
import os
import pickle
import time

import dnnlib
import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm.auto import tqdm

from project_utils.normalization import (
    denormalize_tensor,
    load_norm_stats,
    normalize_tensor,
    resolve_norm_stats_path,
)
from project_utils.paths import PROJECT_ROOT, dataset_trajectory_dir
from project_utils.yaml_simple import load_simple_yaml
from turbulence.lorenz96 import Lorenz96Params, Lorenz96Solver


### Utilities

**Shape convention:** `C=1`, sequence length `K`, stored as `(1, K, 1)`.


In [ ]:
class StackedRandomGenerator:
    def __init__(self, device, seeds):
        self.generators = [
            torch.Generator(device).manual_seed(int(s) % (1 << 32)) for s in seeds
        ]

    def randn(self, size, **kwargs):
        if len(size) < 1 or len(self.generators) != size[0]:
            raise ValueError(
                f"size[0]={size[0] if size else None} must match n_generators={len(self.generators)}"
            )
        return torch.stack(
            [torch.randn(size[1:], generator=g, **kwargs) for g in self.generators]
        )

    def randn_like(self, input):
        return self.randn(
            input.shape, dtype=input.dtype, layout=input.layout, device=input.device
        )


def frame_tensor(x: torch.Tensor, k: int) -> torch.Tensor:
    return x.reshape(1, k, 1)


def make_initial_condition(k: int, F: float, *, device, dtype: torch.dtype) -> torch.Tensor:
    x = torch.full((k,), float(F), device=device, dtype=dtype)
    x[0] = x[0] + torch.tensor(0.01, device=device, dtype=dtype)
    return x


def build_condition_from_history(frames_phys, history, norm_stats):
    if isinstance(frames_phys, torch.Tensor):
        recent = frames_phys[-history:]
        normed = [normalize_tensor(recent[i], norm_stats) for i in range(recent.shape[0])]
        block = torch.stack(normed, dim=0)
    else:
        recent = frames_phys[-history:]
        normed = [normalize_tensor(f, norm_stats) for f in recent]
        block = torch.stack(normed, dim=0)
    C, H, W = block.shape[1], block.shape[2], block.shape[3]
    return block.reshape(history * C, H, W).unsqueeze(0)


def build_condition_from_history_batch(histories, history, norm_stats):
    return torch.cat(
        [build_condition_from_history(h, history, norm_stats) for h in histories],
        dim=0,
    )


def perturb_state(frame, std, seed):
    g = torch.Generator().manual_seed(int(seed) % (1 << 32))
    return frame + std * torch.randn(frame.shape, generator=g, dtype=frame.dtype, device=frame.device)


def ensemble_last_frames(histories):
    return torch.stack([h[-1] for h in histories], dim=0)


def ensemble_mean_rmse(histories, x_truth):
    ens = ensemble_last_frames(histories)
    mean = ens.mean(dim=0)
    spread = ens.std(dim=0, correction=0).mean().item() if ens.shape[0] > 1 else 0.0
    rmse = torch.sqrt(torch.mean((mean - x_truth) ** 2)).item()
    return rmse, spread


def to_seq(traj: torch.Tensor) -> torch.Tensor:
    return traj[:, 0, :, 0]


In [ ]:
# --- User configuration ---
L96_CONFIG = os.path.join(PROJECT_ROOT, "configs/turbulence/lorenz96_default.yaml")
DATA_ROOT = "LORENZ96"
TRAJECTORY_SUBDIR = "full_path"

# NETWORK_PKL = os.path.join(
#     PROJECT_ROOT,
#     "outputs",
#     "00000-lorenz96_path_next-l96_unet1d_tiny-gpus1-batch256-fp32",
#     "network-snapshot-002000.pkl",
# )
NETWORK_PKL = "/home/neil/glocal-condiff-turb/outputs/00003-lorenz96_path_next-l96_unet1d_tiny-gpus1-batch256-fp32/network-snapshot-002000.pkl"


DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
SEED = 0
SEED_DM = 0

K = 1000
T_BURN_IN = 2000
N_DA_CYCLES = 50
OUTPUT_INTERVAL = 0.01

HISTORY = 1

NE = 10
ENSEMBLE_PERTURB_STD = 0.05

device = torch.device(DEVICE)
trajectory_dir = dataset_trajectory_dir(DATA_ROOT, TRAJECTORY_SUBDIR)
norm_stats_path = resolve_norm_stats_path(trajectory_dir, None)


## Phase 1: True L96 burn-in

In [ ]:
cfg = load_simple_yaml(L96_CONFIG)
l96_cfg = dict(cfg.get("lorenz96", {}))
l96_cfg["device"] = str(device)

params = Lorenz96Params(K=K, **l96_cfg)
params.validate()
timesteps_per_output = max(1, int(OUTPUT_INTERVAL / params.dt))
dtype = torch.float32 if params.precision == "single" else torch.float64

x0 = make_initial_condition(K, params.F, device=device, dtype=dtype)
solver = Lorenz96Solver(x0, **params.to_core_kwargs())

burn_frames = [frame_tensor(solver.fields()["x"].detach().cpu(), K)]
for _ in tqdm(range(T_BURN_IN), desc="L96 burn-in"):
    solver.step(timesteps_per_output)
    burn_frames.append(frame_tensor(solver.fields()["x"].detach().cpu(), K))

state_after_burn = {k: v.clone() for k, v in solver.get_state().items()}
path_burn = torch.stack(burn_frames, dim=0)
print("Burn-in frames:", tuple(path_burn.shape))


## Phase 2: Load trained conditional diffusion model + norm stats

In [ ]:
if not os.path.isfile(NETWORK_PKL):
    raise FileNotFoundError(
        f"Set NETWORK_PKL. Not found: {NETWORK_PKL}\n"
        "Train with: torchrun scripts/train/train_cond.py runs=lorenz96_path_next"
    )

norm_stats = load_norm_stats(norm_stats_path)
with dnnlib.util.open_url(NETWORK_PKL, verbose=True) as f:
    net = pickle.load(f)["ema"].to(device).eval()

n_target = 1
n_cond = HISTORY * n_target
assert net.img_input_channels == n_target
assert net.img_cond_channels == n_cond
assert net.img_res_h == K and net.img_res_w == 1

print(f"Loaded {NETWORK_PKL}")
print(f"Model: {net.model_name}")


### Observation operator (sparse direct state)

Likelihood guidance is applied inside `edm_sampler_obs` when `obs["y_obs"]` is set.


In [ ]:
def flat_index(i, K, channel=0):
    return int(channel) * int(K) + int(i)


def random_obs_flat_indices(K, obs_count, obs_channels, seed=0, device="cpu"):
    rng = np.random.default_rng(seed)
    all_idx = []
    for c in obs_channels:
        base = int(c) * int(K)
        all_idx.extend(range(base, base + int(K)))
    flat_idx = rng.choice(np.array(all_idx, dtype=np.int64), size=obs_count, replace=False)
    return torch.tensor(flat_idx, dtype=torch.long, device=device)


def obs_mask_vector(flat_idx, total_size, device=None, dtype=torch.float32):
    dev = device if device is not None else flat_idx.device
    m = torch.zeros(total_size, device=dev, dtype=dtype)
    m[flat_idx.to(dev)] = 1.0
    return m


def observe_state(x_phys, flat_idx, obs_sigma, generator=None):
    flat = x_phys.reshape(-1)
    idx = flat_idx.to(flat.device)
    y = flat[idx].to(torch.float32)
    if obs_sigma > 0:
        y = y + obs_sigma * torch.randn(y.shape, device=y.device, generator=generator)
    return y


def observation_rmse(x_phys, y_obs, flat_idx):
    flat = x_phys.reshape(-1)
    idx = flat_idx.cpu()
    return torch.sqrt(torch.mean((flat[idx] - y_obs.reshape(-1).cpu()) ** 2)).item()


### Observation settings

Uncomment **one** mask block below.

In [ ]:
obs_sigma = 0.5
obs_count = 500  # target number of sites (random / every_nth / mask_1d)

# --- random ---
# obs_method = "random"
# obs_channels = [0]
# obs_flat_idx = random_obs_flat_indices(
#     K, obs_count, obs_channels, seed=SEED + 99, device=device
# )

# --- every_nth: every step-th ring index (~obs_count sites) ---
obs_method = "every_nth"
step = max(1, K // obs_count)
idx = list(range(0, K, step))
obs_flat_idx = torch.tensor(idx, dtype=torch.long, device=device)

# --- indices ---
# obs_method = "indices"
# idx = [0, K // 4, K // 2, 3 * K // 4, K - 1]
# obs_flat_idx = torch.tensor(idx, dtype=torch.long, device=device)

# --- mask_1d ---
# obs_method = "mask_1d"
# mask = torch.zeros(K, dtype=torch.bool)
# mask[:: max(1, K // obs_count)] = True
# obs_flat_idx = torch.nonzero(mask, as_tuple=False).squeeze(1).to(device)

obs_mask = obs_mask_vector(obs_flat_idx, K).reshape(1, K, 1)

fig, ax = plt.subplots(figsize=(10, 2.5))
ax.scatter(
    torch.nonzero(obs_mask[0, :, 0], as_tuple=False).squeeze(1).cpu().numpy(),
    np.ones(obs_flat_idx.numel()),
    s=12,
)
ax.set_xlim(0, K - 1)
ax.set_xlabel("dimension index i")
ax.set_title(f"observation sites ({obs_method}), n_obs={obs_flat_idx.numel()}, step={step if obs_method=='every_nth' else 'n/a'}")
ax.set_yticks([])
plt.tight_layout()
plt.show()

print(
    f"method={obs_method}, n_obs={obs_flat_idx.numel()}, obs_sigma={obs_sigma}, "
    f"mask sum={obs_mask.sum().item():.0f}"
)


### EDM sampling (`edm_sampler_obs`)

Non-gradient observation guidance (no backprop through denoiser).


In [ ]:
NUM_STEPS = 18
GUIDANCE_SCALE = 0.0001
SIGMA_DATA = 0.5
USE_OBS_DENOM = True


def phys_scale_at_flat_idx(flat_idx, norm_stats, seq_len, n_channels, device, dtype=torch.float64):
    flat_idx = flat_idx.to(device)
    block = int(seq_len)
    channels = flat_idx // block
    if (channels < 0).any() or (channels >= n_channels).any():
        raise ValueError(
            f"flat_idx out of range for (C={n_channels}, K={seq_len}) channel-major layout"
        )
    if norm_stats.get("scope", "global") == "global":
        s = 0.5 * float(norm_stats["vmax"] - norm_stats["vmin"])
        return torch.full((flat_idx.numel(),), s, device=device, dtype=dtype)
    ch_scale = 0.5 * (
        norm_stats["vmax_per_channel"] - norm_stats["vmin_per_channel"]
    ).to(device=device, dtype=dtype)
    return ch_scale[channels]


def check_obs_guidance_setup(norm_stats, flat_idx, n_channels, seq_len, device="cpu"):
    if norm_stats.get("scope", "global") == "global":
        lo = float(norm_stats["vmin"])
        hi = float(norm_stats["vmax"])
        x = torch.rand(n_channels, seq_len, 1, device=device) * (hi - lo) + lo
    else:
        vmin_c = norm_stats["vmin_per_channel"].to(device)
        vmax_c = norm_stats["vmax_per_channel"].to(device)
        x = torch.rand(n_channels, seq_len, 1, device=device)
        for c in range(n_channels):
            x[c] = x[c] * float(vmax_c[c] - vmin_c[c]) + float(vmin_c[c])
    err = (denormalize_tensor(normalize_tensor(x, norm_stats), norm_stats) - x).abs().max()
    if float(err) > 1e-4:
        raise RuntimeError(f"norm/denorm round-trip failed: max err {float(err):.2e}")

    idx = flat_idx.to(device)
    if idx.numel() == 0:
        return
    scales = phys_scale_at_flat_idx(idx, norm_stats, seq_len, n_channels, device)
    for k in (0, idx.numel() // 2, idx.numel() - 1):
        ch = int((idx[k] // seq_len).item())
        if norm_stats.get("scope", "global") == "global":
            ref = 0.5 * float(norm_stats["vmax"] - norm_stats["vmin"])
        else:
            ref = float(
                0.5
                * (norm_stats["vmax_per_channel"][ch] - norm_stats["vmin_per_channel"][ch])
            )
        if abs(float(scales[k]) - ref) > 1e-5:
            raise RuntimeError(f"phys_scale mismatch at flat_idx={int(idx[k])}: {scales[k]} vs {ref}")
    print(f"obs guidance checks OK (round-trip err={float(err):.2e})")


def edm_sampler_obs(
    net,
    latents,
    conditions,
    obs=None,
    class_labels=None,
    randn_like=torch.randn_like,
    num_steps=18,
    sigma_min=0.002,
    sigma_max=80,
    rho=7,
    S_churn=0,
    S_min=0,
    S_max=float("inf"),
    S_noise=1,
):
    sigma_min = max(sigma_min, net.sigma_min)
    sigma_max = min(sigma_max, net.sigma_max)

    use_obs = obs is not None and obs.get("y_obs") is not None
    if use_obs:
        device = latents.device
        flat_idx = obs["flat_idx"].to(device)
        obs_sigma = float(obs["obs_sigma"])
        guidance_scale = float(obs.get("guidance_scale", 1.0))
        sigma_data = float(obs.get("sigma_data", SIGMA_DATA))
        use_denom = bool(obs.get("use_denom", False))
        norm_stats = obs["norm_stats"]
        phi = obs.get("phi", lambda u: u)
        phi_prime = obs.get("phi_prime", lambda u: torch.ones_like(u))

        y_obs = obs["y_obs"].to(device, torch.float64)
        if y_obs.dim() == 1:
            y_obs = y_obs.unsqueeze(0)

        n_ch, seq_len, _ = latents.shape[1:]
        std_obs = (
            phys_scale_at_flat_idx(flat_idx, norm_stats, seq_len, n_ch, device)
            if use_denom
            else None
        )

    step_indices = torch.arange(num_steps, dtype=torch.float64, device=latents.device)
    t_steps = (
        sigma_max ** (1 / rho)
        + step_indices / (num_steps - 1) * (sigma_min ** (1 / rho) - sigma_max ** (1 / rho))
    ) ** rho
    t_steps = torch.cat([net.round_sigma(t_steps), torch.zeros_like(t_steps[:1])])

    x_next = latents.to(torch.float64) * t_steps[0]
    neg_log_likelihoods = [] if use_obs else None

    for i, (t_cur, t_next) in enumerate(zip(t_steps[:-1], t_steps[1:])):
        x_cur = x_next
        gamma = min(S_churn / num_steps, np.sqrt(2) - 1) if S_min <= t_cur <= S_max else 0
        t_hat = net.round_sigma(t_cur + gamma * t_cur)
        x_hat = x_cur + (t_hat**2 - t_cur**2).sqrt() * S_noise * randn_like(x_cur)

        guidance = None
        x0_norm_pred = None
        if use_obs:
            with torch.no_grad():
                x0_norm = net(x_hat, t_hat, conditions, class_labels).to(torch.float64)
            x0_leaf = x0_norm.detach().requires_grad_(True)
            x0_phys = denormalize_tensor(x0_leaf, norm_stats)
            x0_flat = x0_phys.reshape(x0_phys.shape[0], -1)

            gx = phi(x0_flat[:, flat_idx])
            resid = gx - y_obs

            if use_denom:
                sigma = float(t_hat.reshape(-1)[0].item())
                r2 = sigma**2 * sigma_data**2 / (sigma**2 + sigma_data**2)
                pp = phi_prime(x0_flat[:, flat_idx].detach())
                denom = obs_sigma**2 + r2 * std_obs.pow(2) * pp.pow(2)
            else:
                denom = obs_sigma**2

            nll = 0.5 * (resid.pow(2) / denom).sum(dim=1)
            log_p = -nll.sum()
            guidance = torch.autograd.grad(log_p, x0_leaf)[0]
            neg_log_likelihoods.append(float(nll.mean().detach().cpu()))
            x0_norm_pred = x0_norm

        with torch.no_grad():
            if x0_norm_pred is None:
                x0_norm_pred = net(x_hat, t_hat, conditions, class_labels).to(torch.float64)
            d_cur = (x_hat - x0_norm_pred) / t_hat
            x_next = x_hat + (t_next - t_hat) * d_cur

            if i < num_steps - 1:
                denoised = net(x_next, t_next, conditions, class_labels).to(torch.float64)
                d_prime = (x_next - denoised) / t_next
                x_next = x_hat + (t_next - t_hat) * (0.5 * d_cur + 0.5 * d_prime)

            if use_obs:
                x_next = x_next + guidance_scale * guidance

    if use_obs:
        return x_next, neg_log_likelihoods
    return x_next


check_obs_guidance_setup(norm_stats, obs_flat_idx, n_target, K, device="cpu")


## Phase 4: DA loop (re-runnable)

In [ ]:
SAVE_RESULTS = False
SAVE_PATH = os.path.join(trajectory_dir, "da_ensf_dm_demo.pt")

core_kw = params.to_core_kwargs()
truth = Lorenz96Solver(path_burn[-1][0, :, 0].to(device=device, dtype=dtype), **core_kw)
truth.set_state(state_after_burn)

n_burn = path_burn.shape[0]
hist_analysis = []
hist_forecast = []
for k in range(NE):
    ha = [path_burn[i].clone() for i in range(n_burn)]
    hf = [path_burn[i].clone() for i in range(n_burn)]
    ha[-1] = perturb_state(ha[-1], ENSEMBLE_PERTURB_STD, SEED + 1000 + k)
    hf[-1] = ha[-1].clone()
    hist_analysis.append(ha)
    hist_forecast.append(hf)

rnd = StackedRandomGenerator(device, [SEED_DM + k for k in range(NE)])

rmse_truth_vs_analysis = []
rmse_truth_vs_forecast = []
spread_analysis = []
spread_forecast = []
obs_rmse = []
neg_log_likelihood_last = []

cycle_times = []
pbar = tqdm(range(N_DA_CYCLES), desc="DA cycles", unit="cycle")
for cycle in pbar:
    t_cycle_start = time.perf_counter()

    truth.step(timesteps_per_output)
    x_truth = frame_tensor(truth.fields()["x"].detach().cpu(), K)
    y_obs = observe_state(x_truth, obs_flat_idx, obs_sigma).to(device)
    if not torch.isfinite(y_obs).all():
        print(f"Stopping at cycle {cycle}: non-finite observations y_obs")
        break
    obs_rmse.append(observation_rmse(x_truth, y_obs, obs_flat_idx))

    cond_a = build_condition_from_history_batch(hist_analysis, HISTORY, norm_stats).to(device)
    cond_f = build_condition_from_history_batch(hist_forecast, HISTORY, norm_stats).to(device)
    if not (torch.isfinite(cond_a).all() and torch.isfinite(cond_f).all()):
        print(f"Stopping at cycle {cycle}: non-finite DM conditions")
        break

    obs_cfg = {
        "y_obs": y_obs,
        "flat_idx": obs_flat_idx,
        "obs_sigma": obs_sigma,
        "norm_stats": norm_stats,
        "guidance_scale": GUIDANCE_SCALE,
        "sigma_data": SIGMA_DATA,
        "use_denom": USE_OBS_DENOM,
    }

    lat_a = rnd.randn([NE, n_target, K, 1], device=device)
    lat_f = rnd.randn([NE, n_target, K, 1], device=device)

    with torch.no_grad():
        sample_f_norm = edm_sampler_obs(net, lat_f, cond_f, num_steps=NUM_STEPS).to(torch.float32)
    if not torch.isfinite(sample_f_norm).all():
        print(f"Stopping at cycle {cycle}: non-finite DM forecast sample")
        break

    next_forecast = denormalize_tensor(sample_f_norm.cpu(), norm_stats)
    sample_a_norm, neg_log_likelihoods = edm_sampler_obs(
        net,
        lat_a,
        cond_a,
        obs=obs_cfg,
        num_steps=NUM_STEPS,
    )
    if not torch.isfinite(sample_a_norm).all():
        print(f"Stopping at cycle {cycle}: non-finite observation-guided DM sample")
        break

    next_analysis = denormalize_tensor(sample_a_norm.cpu().float(), norm_stats)
    if not (torch.isfinite(next_forecast).all() and torch.isfinite(next_analysis).all()):
        print(f"Stopping at cycle {cycle}: non-finite ensemble state")
        break

    for k in range(NE):
        hist_forecast[k].append(next_forecast[k])
        hist_analysis[k].append(next_analysis[k])

    rmse_a, spr_a = ensemble_mean_rmse(hist_analysis, x_truth)
    rmse_f, spr_f = ensemble_mean_rmse(hist_forecast, x_truth)
    if not all(np.isfinite([rmse_a, spr_a, rmse_f, spr_f])):
        for k in range(NE):
            hist_forecast[k].pop()
            hist_analysis[k].pop()
        print(f"Stopping at cycle {cycle}: non-finite RMSE/spread metrics")
        break

    neg_log_likelihood_last.append(
        neg_log_likelihoods[-1] if neg_log_likelihoods else float("nan")
    )
    rmse_truth_vs_analysis.append(rmse_a)
    rmse_truth_vs_forecast.append(rmse_f)
    spread_analysis.append(spr_a)
    spread_forecast.append(spr_f)

    cycle_times.append(time.perf_counter() - t_cycle_start)
    avg_step = sum(cycle_times) / len(cycle_times)
    remaining = avg_step * (N_DA_CYCLES - cycle - 1)
    pbar.set_postfix(step=f"{avg_step:.2f}s", remaining=f"{remaining:.0f}s")

n_da_cycles_completed = len(rmse_truth_vs_analysis)
print(
    f"Completed {n_da_cycles_completed}/{N_DA_CYCLES} DA cycles. "
    f"Last RMSE (ens. mean) analysis {rmse_truth_vs_analysis[-1] if rmse_truth_vs_analysis else float('nan'):.4f}, "
    f"forecast {rmse_truth_vs_forecast[-1] if rmse_truth_vs_forecast else float('nan'):.4f}"
)
if cycle_times:
    print(f"Avg time per cycle: {sum(cycle_times) / len(cycle_times):.2f}s")

truth_frames = [path_burn[i].clone() for i in range(path_burn.shape[0])]
truth_solver_log = Lorenz96Solver(path_burn[-1][0, :, 0].to(device=device, dtype=dtype), **core_kw)
truth_solver_log.set_state(state_after_burn)
for _ in range(n_da_cycles_completed):
    truth_solver_log.step(timesteps_per_output)
    truth_frames.append(frame_tensor(truth_solver_log.fields()["x"].detach().cpu(), K))

path_truth_full = torch.stack(truth_frames, dim=0)
path_analysis_ens = torch.stack([torch.stack(h, dim=0) for h in hist_analysis], dim=0)
path_forecast_ens = torch.stack([torch.stack(h, dim=0) for h in hist_forecast], dim=0)
path_analysis = path_analysis_ens.mean(dim=0)
path_forecast = path_forecast_ens.mean(dim=0)

if SAVE_RESULTS:
    torch.save(
        {
            "NE": NE,
            "n_da_cycles_completed": n_da_cycles_completed,
            "path_truth": path_truth_full,
            "path_analysis": path_analysis,
            "path_forecast": path_forecast,
            "path_analysis_ens": path_analysis_ens,
            "path_forecast_ens": path_forecast_ens,
            "rmse_analysis": rmse_truth_vs_analysis,
            "rmse_forecast": rmse_truth_vs_forecast,
            "spread_analysis": spread_analysis,
            "spread_forecast": spread_forecast,
            "obs_rmse": obs_rmse,
        },
        SAVE_PATH,
    )
    print("Saved", SAVE_PATH, "ensemble shape", tuple(path_analysis_ens.shape))
else:
    print("SAVE_RESULTS=False — results kept in memory only.")


## Phase 5: Metrics vs assimilation cycle

In [ ]:
n_done = len(rmse_truth_vs_analysis)
cycles = np.arange(n_done)
fig, axs = plt.subplots(3, 1, figsize=(9, 8), sharex=True)
if n_done == 0:
    print("No completed DA cycles to plot.")
else:
    axs[0].plot(cycles, rmse_truth_vs_forecast, "o-", label="DM forecast (ens. mean)")
    axs[0].plot(cycles, rmse_truth_vs_analysis, "o-", label="DM + obs guidance (ens. mean)")
    axs[0].set_ylabel("RMSE vs truth")
    axs[0].legend()
    axs[0].grid(True, alpha=0.3)
    axs[1].plot(cycles, spread_forecast, "o-", label="forecast spread")
    axs[1].plot(cycles, spread_analysis, "o-", label="analysis spread")
    axs[1].set_ylabel("ens. spread (mean std)")
    axs[1].legend()
    axs[1].grid(True, alpha=0.3)
    axs[2].plot(cycles, obs_rmse[:n_done], "o-", color="gray", label="obs error")
    axs[2].set_ylabel("obs RMSE")
    axs[2].set_xlabel("assimilation cycle (after burn-in)")
    axs[2].legend()
    axs[2].grid(True, alpha=0.3)
    plt.suptitle(f"L96 DA with NE={NE}: {n_done} cycles completed")
    plt.tight_layout()
    plt.show()


## Ring snapshots: truth vs ensemble-mean forecast / analysis

In [ ]:
t0 = path_burn.shape[0] - 1
truth_seg = to_seq(path_truth_full[t0:])
analysis_seg = to_seq(path_analysis[t0:])
forecast_seg = to_seq(path_forecast[t0:])
k_show = min(10, truth_seg.shape[0] - 1)
x_axis = np.arange(K)

fig, axs = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
axs[0].plot(x_axis, truth_seg[k_show].numpy(), color="C0")
axs[0].set_title(f"truth (L96 solver), k={k_show}")
axs[1].plot(x_axis, analysis_seg[k_show].numpy(), color="C1")
axs[1].set_title(f"DM + obs guidance analysis (ens. mean), k={k_show}")
axs[2].plot(x_axis, forecast_seg[k_show].numpy(), color="C2")
axs[2].set_title(f"DM forecast (ens. mean), k={k_show}")
for ax in axs:
    ax.set_ylabel("x")
    ax.grid(True, alpha=0.3)
axs[-1].set_xlabel("dimension index i")
plt.tight_layout()
plt.show()


## Animate: selected ensemble members

Saves animation to HTML file instead of embedding huge output in the notebook (prevents file corruption).


In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

ANIM_ENS_MEMBERS = [0, 1, 2]
ANIM_MAX_FRAMES = 80
ANIM_OUT = os.path.join(trajectory_dir, "da_ensf_ring_compare.html")

t0 = path_burn.shape[0] - 1
truth_seg = to_seq(path_truth_full[t0:])
T_seg = min(
    truth_seg.shape[0],
    path_analysis_ens.shape[1] - t0,
    path_forecast_ens.shape[1] - t0,
    ANIM_MAX_FRAMES,
)
n_rows = len(ANIM_ENS_MEMBERS)
x_axis = np.arange(K)

stacked = [truth_seg[:T_seg]]
for k in ANIM_ENS_MEMBERS:
    stacked.append(to_seq(path_analysis_ens[k, t0 : t0 + T_seg]))
    stacked.append(to_seq(path_forecast_ens[k, t0 : t0 + T_seg]))
stacked = torch.stack(stacked, dim=0)
vmin, vmax = float(stacked.min()), float(stacked.max())

fig, axs = plt.subplots(n_rows, 3, figsize=(10, 2.8 * n_rows), squeeze=False)
fig.subplots_adjust(hspace=0.35, wspace=0.12)
ims = []
for r, k in enumerate(ANIM_ENS_MEMBERS):
    row_data = [
        truth_seg[:T_seg],
        to_seq(path_analysis_ens[k, t0 : t0 + T_seg]),
        to_seq(path_forecast_ens[k, t0 : t0 + T_seg]),
    ]
    col_labels = ["truth", f"analysis ens {k}", f"forecast ens {k}"]
    for c in range(3):
        ax = axs[r, c]
        (line,) = ax.plot(x_axis, row_data[c][0].numpy(), linewidth=1.0)
        ims.append((row_data[c], line))
        ax.set_ylim(vmin, vmax)
        ax.set_title(col_labels[c])
        ax.grid(True, alpha=0.3)
        if r == n_rows - 1:
            ax.set_xlabel("dimension i")
title = fig.suptitle("")


def update(frame_k):
    for row_data, line in ims:
        line.set_ydata(row_data[frame_k].numpy())
    title.set_text(f"post-burn-in k={frame_k}  |  cols: truth / analysis / forecast")
    return [line for _, line in ims] + [title]


anim = FuncAnimation(fig, update, frames=T_seg, interval=120, blit=True, repeat=True)
anim.save(ANIM_OUT, writer="html")
plt.close(fig)
print("Saved animation:", ANIM_OUT)
display(HTML(f'<a href="{ANIM_OUT}" target="_blank">Open ring comparison animation</a>'))
